# 01 - Conditional GAN (cGAN) Training
A cGAN learns `G(z, y)` and `D(x, y)`:
* **Generator** concatenates the latent vector `z` with a learned class embedding and outputs a `1x28x28` image through `Tanh` (range `[-1, 1]`).
* **Discriminator** receives the image *and* the label (label embedding reshaped to a 28x28 map, concatenated as a second channel) and outputs `P(real)`.

**Objective (standard minimax GAN with BCE):**
$$\min_G \max_D \; \mathbb{E}_{x,y}[\log D(x,y)] + \mathbb{E}_{z,y}[\log(1 - D(G(z,y),y))]$$
Implemented with `nn.BCELoss`: the discriminator minimises the *average* of BCE(real, 1) and BCE(fake, 0); the generator uses the non-saturating form BCE(D(G(z,y),y), 1).

In [ ]:
import os, sys, random
import numpy as np
import torch
import matplotlib.pyplot as plt

# Set QUICK_RUN=1 in the environment for a tiny smoke-test run (few samples / 1 epoch). Leave unset for the real run.
QUICK_RUN = os.environ.get("QUICK_RUN", "0") == "1"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| QUICK_RUN:", QUICK_RUN)
os.makedirs("checkpoints", exist_ok=True); os.makedirs("outputs", exist_ok=True)

In [ ]:
import torch.nn as nn
from data_utils.dataloader import get_dataloader
from model import Generator, Discriminator
from training import train_cgan
from utils.visualize import show_class_grid, plot_losses, show_interpolation

# ---------------- configuration ----------------
LATENT_DIM = 100
NUM_CLASSES = 10
BATCH_SIZE = 128
EPOCHS = 1 if QUICK_RUN else 50
LR = 2e-4
SUBSET = 1024 if QUICK_RUN else None

train_loader = get_dataloader(batch_size=BATCH_SIZE, train=True, subset=SUBSET)
print("batches per epoch:", len(train_loader))

### Build the models

In [ ]:
G = Generator(latent_dim=LATENT_DIM, num_classes=NUM_CLASSES).to(device)
D = Discriminator(num_classes=NUM_CLASSES).to(device)
count = lambda m: sum(p.numel() for p in m.parameters())
print(f"Generator params: {count(G):,} | Discriminator params: {count(D):,}")

# sanity check of shapes / range
z = torch.randn(4, LATENT_DIM, device=device); y = torch.arange(4, device=device)
fake = G(z, y)
print("G output:", tuple(fake.shape), f"range [{fake.min():.2f}, {fake.max():.2f}]", "| D output:", tuple(D(fake, y).shape))
assert fake.shape == (4, 1, 28, 28)

### Train (adversarial loop: separate Discriminator and Generator updates)

In [ ]:
criterion = nn.BCELoss()   # standard GAN minimax objective

# fixed noise/labels -> watch the same samples improve over training
fixed_z = torch.randn(100, LATENT_DIM, device=device)
fixed_labels = torch.arange(10, device=device).repeat_interleave(10)
snapshots = {}
def snapshot(epoch, gen):
    if epoch in (1, 5, 20, EPOCHS) or QUICK_RUN:
        gen.eval()
        with torch.no_grad():
            snapshots[epoch] = gen(fixed_z, fixed_labels).cpu()
        gen.train()

history = train_cgan(G, D, train_loader, epochs=EPOCHS, device=device, latent_dim=LATENT_DIM,
                     num_classes=NUM_CLASSES, lr=LR, criterion=criterion,
                     checkpoint_dir="checkpoints", save_every=10, epoch_callback=snapshot)

### Training curves

In [ ]:
plot_losses({"D loss": history["d_loss"], "G loss": history["g_loss"]}, "cGAN losses", "outputs/01_cgan_losses.png")
plot_losses({"D(x,y) on real": history["d_real"], "D(G(z,y),y) on fake": history["d_fake"]}, "Discriminator outputs")

### Generated digits 0-9 (each row = one class label)

In [ ]:
G.eval()
with torch.no_grad():
    samples = G(fixed_z, fixed_labels).cpu()
show_class_grid(samples, fixed_labels.cpu(), per_class=10,
                title="cGAN samples - rows are labels 0 (top) to 9 (bottom)", save_path="outputs/01_cgan_class_grid.png")

### Diversity check (must pass before moving on)
Same label, 100 different noise vectors, generator in `eval()` mode (the mode used to build the dataset). If the *variation across z* is far below the real-data value, the generator is ignoring `z` (mode collapse).

In [ ]:
from data_utils.dataloader import dataset_to_tensors
from utils.visualize import show_images
real_x_all, real_y_all = dataset_to_tensors(train=True)
G.eval()
with torch.no_grad():
    x3 = G(torch.randn(100, LATENT_DIM, device=device), torch.full((100,), 3, device=device)).cpu()
real_var = real_x_all[real_y_all == 3][:1000].std(0).mean().item()
print(f"variation across z, digit 3 (eval mode): {x3.std(0).mean().item():.3f}   | real digit-3 images: {real_var:.3f}")
show_images(x3[:40], nrow=10, title="40 different z, label 3 (should look like different handwriting styles of '3')")

### Progress over training (same noise + labels)

In [ ]:
for ep, imgs in sorted(snapshots.items()):
    show_class_grid(imgs, fixed_labels.cpu(), per_class=10, title=f"cGAN after epoch {ep}")

### Stand-out: latent-space walk
Interpolate linearly between two noise vectors while keeping the label fixed -> smooth transition between two different *styles* of the same digit.

In [ ]:
STEPS = 10
rows = []
with torch.no_grad():
    for digit in [3, 7, 8]:
        z0, z1 = torch.randn(1, LATENT_DIM, device=device), torch.randn(1, LATENT_DIM, device=device)
        alphas = torch.linspace(0, 1, STEPS, device=device).view(-1, 1)
        zs = (1 - alphas) * z0 + alphas * z1
        rows.append(G(zs, torch.full((STEPS,), digit, device=device)).cpu())
show_interpolation(torch.stack(rows), title="Latent interpolation (digits 3, 7, 8)", save_path="outputs/01_latent_walk.png")

# interpolate between two different CLASSES with the same z
with torch.no_grad():
    z0 = torch.randn(1, LATENT_DIM, device=device).repeat(10, 1)
    show_interpolation(G(z0, torch.arange(10, device=device)).cpu(), title="Same z, labels 0-9 (style is shared)")

### Notes
* A healthy run keeps `D(x,y)` around 0.6-0.8 and `D(G(z,y),y)` around 0.2-0.4; a collapse of `D` to 0/1 would indicate training failure.
* Weights are saved to `checkpoints/cgan_final.pt` (used by notebook 03).